# Train a PyTorch Regression Model with Ray Train + Tune on Backblaze B2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/backblaze-b2-samples/notebooks/blob/main/ray-train-tune-checkpoints/ray_train_b2.ipynb) [![Open In Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/backblaze-b2-samples/notebooks/HEAD?urlpath=lab/tree/ray-train-tune-checkpoints/ray_train_b2.ipynb) [![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/backblaze-b2-samples/notebooks?quickstart=1)

Learn how to use **[Ray Train](https://docs.ray.io/en/latest/train/train.html)** and **[Ray Tune](https://docs.ray.io/en/latest/tune/index.html)** with the **California Housing dataset** hosted in **[Backblaze B2](https://www.backblaze.com/b2/cloud-storage.html)** cloud storage. You'll train a small PyTorch regression model that predicts median house prices from eight neighborhood features, and write every checkpoint and Tune-trial artifact straight back to B2.

**What you'll build:** A distributed PyTorch training pipeline that streams parquet data from a public B2 bucket, runs a hyperparameter sweep with Ray Tune, and persists every checkpoint to *your* B2 bucket, all configured with three environment variables.

## About Ray Train + Ray Tune

**Ray Train** is the distributed training library in the [Ray](https://www.ray.io/) ecosystem. It wraps your existing PyTorch (or Lightning, JAX, TensorFlow…) training loop with a few lines and gives you data-parallel scaling, automatic checkpoint sync to remote storage, and fault tolerance. **Ray Tune** layers hyperparameter sweeps on top: same training function, multiple trials in parallel, results aggregated.

Both libraries route checkpoints and trial artifacts through a single ``RunConfig.storage_path``. For ``s3://`` URIs, that goes through pyarrow's ``S3FileSystem``, which is exactly how Backblaze B2 plugs in.

## About California Housing

**California Housing** is a classic regression dataset bundled with scikit-learn:

- **Rows:** 20,640 (16,512 train + 4,128 test in the public B2 mirror)
- **Features:** 8 numeric features (income, age, rooms, location, …)
- **Target:** Median house value in $100,000 units
- **Use cases:** Regression baselines, hyperparameter-search benchmarks, ML education

## Dataset Hosted on Backblaze B2

This dataset is **publicly accessible** from Backblaze B2 cloud storage:

- **Location:** ``s3://b2datasets/ray-train-demo/california_housing/``
- **Endpoint:** ``https://s3.us-west-001.backblazeb2.com``
- **Access:** Anonymous read for the parquet files, no API key needed for §2 of this notebook
- **Format:** Two parquet files (``train.parquet``, ``test.parquet``)
- **Benefits:** Stream parquet directly to your training environment without manual downloads or local storage

**Note on B2 Public Buckets:** Files in public B2 buckets can be downloaded over plain HTTPS without authentication, and pyarrow's ``S3FileSystem(anonymous=True)`` reads them too, no API key required for §2.

## What This Notebook Does

1. **Configure Backblaze B2 via three env vars**, endpoint + AWS-named credentials. Pulls credentials from GitHub Codespaces secrets, Google Colab Secrets, or an interactive prompt depending on where you run.
2. **Stream parquet from the public B2 bucket** as a Ray Data ``Dataset``, both via the S3 protocol and via plain HTTPS for ad-hoc inspection.
3. **Train a small PyTorch regression model with Ray Train**, distributed across two workers, with one checkpoint per epoch.
4. **Run an optional Ray Tune sweep** over learning rates, three trials in parallel, results aggregated.
5. **Save checkpoints to *your* B2 bucket**, via ``RunConfig(storage_path="s3://your-private-bucket/...")``.


## Setup

Install Ray Train + Tune, PyTorch (used by ``TorchTrainer``), pandas + pyarrow (used by Ray Data and the HTTPS demo), and scikit-learn (used by ``prepare_dataset.py``):


In [ ]:
%pip install -q pandas pyarrow "ray[train,tune]" scikit-learn torch


## Configuration

### Reading from the Public Dataset Bucket

Three env vars wire B2 into Ray's S3 storage path. Set them once and every ``s3://`` URI in this process automatically routes to B2:

| Env var | Set to |
|---|---|
| ``AWS_ENDPOINT_URL_S3`` | endpoint for the **B2 region your private output bucket lives in**. Defaults to ``https://s3.us-west-001.backblazeb2.com`` (where the public dataset lives). If your output bucket is in a different region (``us-west-004``, ``eu-central-003``, etc.), export this env var with that region's endpoint **before** running this notebook. |
| ``AWS_ACCESS_KEY_ID`` | your B2 application key ID (only needed for the §3 write path) |
| ``AWS_SECRET_ACCESS_KEY`` | your B2 application key (only needed for the §3 write path) |

The cell below uses ``os.environ.setdefault(...)`` so any endpoint you have already exported wins. The §2 read path against the public dataset bucket always targets ``us-west-001`` explicitly, so you can override ``AWS_ENDPOINT_URL_S3`` for your private region without breaking the public read.

For the §2 read path against the public dataset bucket, no credentials are needed, we use ``S3FileSystem(anonymous=True)``.

In [ ]:
import os

# The public dataset for §2 lives in us-west-001. We pin this endpoint
# explicitly when constructing the public-read S3FileSystem below, so the
# §2 read path works regardless of any AWS_ENDPOINT_URL_S3 override.
PUBLIC_DATASET_ENDPOINT = "https://s3.us-west-001.backblazeb2.com"

# pyarrow's S3 SDK reads AWS_ENDPOINT_URL_S3 once at URI-resolution time
# (pyarrow.fs.FileSystem.from_uri), so it must be set BEFORE the first Ray
# Train / Ray Data call that touches an s3:// URI. Use setdefault so that
# any endpoint already exported (e.g., for a private bucket in a different
# B2 region) wins.
os.environ.setdefault("AWS_ENDPOINT_URL_S3", PUBLIC_DATASET_ENDPOINT)

# B2 buckets are region-locked. Without an explicit region the SDK defaults
# to us-east-1 and B2 returns ACCESS_DENIED on the SigV4 signature check.
os.environ.setdefault("AWS_DEFAULT_REGION", "us-west-001")
os.environ.setdefault("AWS_REGION", "us-west-001")

# Recommended outside AWS to skip a 1-second EC2 IMDS lookup:
os.environ.setdefault("AWS_EC2_METADATA_DISABLED", "true")

B2_ENDPOINT = os.environ["AWS_ENDPOINT_URL_S3"]
PUBLIC_BUCKET = "b2datasets"
DATASET_PREFIX = "ray-train-demo/california_housing"

print(f"Public-read endpoint:  {PUBLIC_DATASET_ENDPOINT}")
print(f"Write endpoint:        {B2_ENDPOINT}")

### Your B2 Bucket for Outputs

The §3 write path needs ``AWS_ACCESS_KEY_ID`` + ``AWS_SECRET_ACCESS_KEY`` for an application key with write access to your bucket. Pick the path that matches where you're running this notebook:

| Environment | Secret store | Notes |
|---|---|---|
| **GitHub Codespaces** ⭐ | User Settings → Codespaces → *Codespaces secrets* | Add the secrets at <https://github.com/settings/codespaces>, scope them to ``backblaze-b2-samples/notebooks``. They're injected as env vars natively, cleanest setup. |
| **Google Colab** | Left sidebar → 🔑 *Secrets* | Add the two secrets and grant this notebook access. The cell below pulls them via ``google.colab.userdata.get(...)``. |
| **Binder / local** | None / shell env | The cell below falls back to ``getpass.getpass()`` to prompt at runtime. The §2 read path needs no credentials, so you can also just skip §3, the §3 cells will fall back to a local tmpdir for checkpoints. |

**Get your B2 credentials:** Log into [Backblaze B2](https://www.backblaze.com/b2/cloud-storage.html), create a private bucket (or use an existing one) for checkpoints, and create an Application Key with write access scoped to that bucket. The Key ID and Application Key go into the env vars above.

The cell below tries each source in priority order: explicit AWS-named env vars (already exported / Codespaces injection) → Backblaze-named ``B2_APPLICATION_KEY_ID`` / ``B2_APPLICATION_KEY`` env vars → Google Colab Secrets → interactive ``getpass`` prompt.

In [ ]:
def _resolve_credential(env_var: str, *, b2_alias: str | None = None, prompt: str = "") -> None:
    """Populate env_var from (in priority order): existing env, Backblaze-named alias,
    Colab Secrets, then an interactive getpass prompt."""
    if os.environ.get(env_var):
        return  # already set in shell or by Codespaces
    if b2_alias and os.environ.get(b2_alias):
        os.environ[env_var] = os.environ[b2_alias]
        return
    try:
        from google.colab import userdata  # type: ignore[import-not-found]
        try:
            os.environ[env_var] = userdata.get(env_var)
            return
        except Exception:
            pass  # secret not configured in Colab, fall through to getpass
    except ImportError:
        pass  # not running in Colab
    if prompt:
        import getpass
        os.environ[env_var] = getpass.getpass(prompt)


_resolve_credential(
    "AWS_ACCESS_KEY_ID",
    b2_alias="B2_APPLICATION_KEY_ID",
    prompt="B2 application key ID (or press Enter to skip §3): ",
)
_resolve_credential(
    "AWS_SECRET_ACCESS_KEY",
    b2_alias="B2_APPLICATION_KEY",
    prompt="B2 application key (or press Enter to skip §3): ",
)

# CI / non-interactive use: PRIVATE_B2_BUCKET in env wins over the prompt.
# Same for the storage prefix; defaults to "ray-train-runs" if unset.
YOUR_BUCKET = os.environ.get("PRIVATE_B2_BUCKET", "").strip()
if not YOUR_BUCKET:
    YOUR_BUCKET = input("Your private B2 bucket name (for checkpoints, or Enter to skip §3): ").strip()
STORAGE_PREFIX = os.environ.get("PRIVATE_B2_PREFIX", "ray-train-runs").strip("/")

have_creds = bool(os.environ.get("AWS_ACCESS_KEY_ID") and os.environ.get("AWS_SECRET_ACCESS_KEY"))

# Decide where Ray Train will write checkpoints. If the user supplied both
# credentials and a bucket name, write to B2; otherwise fall back to a local
# tmpdir so §3 still runs end-to-end and demonstrates the training loop.
import tempfile

if have_creds and YOUR_BUCKET:
    STORAGE_PATH = f"s3://{YOUR_BUCKET}/{STORAGE_PREFIX}"
    STORAGE_KIND = "B2"
else:
    STORAGE_PATH = tempfile.mkdtemp(prefix="ray-train-demo-")
    STORAGE_KIND = "local"

print(f"\nEndpoint:                {B2_ENDPOINT}")
print(f"Public read bucket:      {PUBLIC_BUCKET}/{DATASET_PREFIX}/")
print(f"§3 write path enabled:   {have_creds and bool(YOUR_BUCKET)}")
print(f"§3 storage_path:         {STORAGE_PATH}  ({STORAGE_KIND})")
if STORAGE_KIND == "local":
    print("\n  (No B2 credentials/bucket, checkpoints will land in the local tmpdir above.")
    print("   Re-run §1 with credentials + a bucket name to write checkpoints to B2.)")


## Stream the Training Data from B2

The public dataset bucket is anonymously readable, so we construct an explicit ``S3FileSystem(anonymous=True, endpoint_override=...)`` for the read path and hand it to ``ray.data.read_parquet``. Ray Data will lazily stream parquet from B2 as the trainer iterates batches, no full local download.


In [ ]:
import pyarrow.fs

import ray

# Pin the public-read filesystem to the us-west-001 endpoint where the public
# dataset actually lives, regardless of how AWS_ENDPOINT_URL_S3 is set.
public_fs = pyarrow.fs.S3FileSystem(
    endpoint_override=PUBLIC_DATASET_ENDPOINT,
    anonymous=True,
)

train_ds = ray.data.read_parquet(
    f"{PUBLIC_BUCKET}/{DATASET_PREFIX}/train.parquet",
    filesystem=public_fs,
)
test_ds = ray.data.read_parquet(
    f"{PUBLIC_BUCKET}/{DATASET_PREFIX}/test.parquet",
    filesystem=public_fs,
)

print(f"Train rows: {train_ds.count():,}")
print(f"Test rows:  {test_ds.count():,}")
print("\nSchema:", train_ds.schema())

### Alternative: Plain HTTPS Download

Because the dataset bucket is configured as **Public** in B2, its files are also reachable via plain HTTPS, no S3 SDK, no application key, no ``S3FileSystem`` construction. Useful for ad-hoc inspection (``curl``, ``wget``, browser) or when handing a parquet file directly to ``pandas`` / ``polars`` / ``duckdb``:


In [ ]:
import pandas as pd

PUBLIC_HTTPS_URL = (
    f"https://{PUBLIC_BUCKET}.s3.us-west-001.backblazeb2.com/"
    f"{DATASET_PREFIX}/train.parquet"
)
print(f"Friendly subdomain URL: {PUBLIC_HTTPS_URL}\n")

train_pdf = pd.read_parquet(PUBLIC_HTTPS_URL)
print(f"Pandas-loaded shape: {train_pdf.shape}")
train_pdf.head(3)


## Define the Model

A small three-layer MLP that maps the eight California Housing features to a single regression output (median house value). Small enough to train in a few minutes on CPU; bigger problems would scale this up and add ``ScalingConfig(use_gpu=True)``.


In [ ]:
import torch
import torch.nn as nn

FEATURE_COLUMNS = [
    "MedInc", "HouseAge", "AveRooms", "AveBedrms",
    "Population", "AveOccup", "Latitude", "Longitude",
]
TARGET_COLUMN = "MedHouseVal"


def build_model() -> nn.Module:
    return nn.Sequential(
        nn.Linear(len(FEATURE_COLUMNS), 64), nn.ReLU(),
        nn.Linear(64, 32), nn.ReLU(),
        nn.Linear(32, 1),
    )


print(build_model())


## Define the Training Function

The ``train_func`` runs once per Ray Train worker. It pulls its dataset shard via ``ray.train.get_dataset_shard()``, runs a standard PyTorch loop, and reports metrics + checkpoints via ``ray.train.report()``.

Each ``ray.train.report(checkpoint=...)`` call uploads the checkpoint to ``RunConfig.storage_path`` (which we'll point at your B2 bucket in the next step).


In [ ]:
import tempfile

import ray.train
from ray.train import Checkpoint


def train_func(config):
    lr = config.get("lr", 0.01)
    epochs = config.get("epochs", 5)
    batch_size = config.get("batch_size", 256)

    model = ray.train.torch.prepare_model(build_model())
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    train_shard = ray.train.get_dataset_shard("train")

    for epoch in range(epochs):
        model.train()
        running_loss, batches = 0.0, 0
        for batch in train_shard.iter_torch_batches(batch_size=batch_size):
            features = torch.stack([batch[col] for col in FEATURE_COLUMNS], dim=1).float()
            target = batch[TARGET_COLUMN].float().unsqueeze(1)
            preds = model(features)
            loss = loss_fn(preds, target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += float(loss.item())
            batches += 1

        # Save a checkpoint each epoch. Ray Train uploads the checkpoint to
        # whatever RunConfig.storage_path resolves to: a local directory, or
        # an s3:// URI that pyarrow routes to B2 via AWS_ENDPOINT_URL_S3.
        with tempfile.TemporaryDirectory() as tmp:
            torch.save(model.state_dict(), os.path.join(tmp, "model.pt"))
            ray.train.report(
                {"loss": running_loss / max(batches, 1), "epoch": epoch},
                checkpoint=Checkpoint.from_directory(tmp),
            )

## Train the Model + Save Checkpoints

If you supplied credentials and a bucket name in §1, ``STORAGE_PATH`` points at ``s3://<your-bucket>/ray-train-runs`` and Ray Train will upload checkpoints to B2 (pyarrow's URI resolver picks up the endpoint and credentials from env vars).

If you skipped §1, ``STORAGE_PATH`` is a local temp directory and the cells below still run end-to-end, same training loop, checkpoints just land on disk instead of in B2. Re-run §1 with credentials + a private bucket to switch over.

In [ ]:
from ray.train import RunConfig, ScalingConfig
from ray.train.torch import TorchTrainer

trainer = TorchTrainer(
    train_func,
    scaling_config=ScalingConfig(num_workers=2, use_gpu=False),
    train_loop_config={"lr": 0.01, "epochs": 5, "batch_size": 256},
    datasets={"train": train_ds},
    run_config=RunConfig(
        name="california_housing_b2_demo",
        storage_path=STORAGE_PATH,
    ),
)
result = trainer.fit()

print(f"\n✓ Training complete (storage: {STORAGE_KIND})")
print(f"  Final loss:      {result.metrics['loss']:.4f}")
print(f"  Checkpoint URI:  {result.checkpoint.path}")

## (Optional) Hyperparameter Sweep with Ray Tune

Wrap the same ``TorchTrainer`` in a ``Tuner`` to sweep across learning rates. Each trial gets its own subdirectory under ``storage_path`` (B2 or local, whichever §1 picked), so all artifacts land together.

In [ ]:
import os

# In CI we skip the optional Tune sweep because (a) it triples B2 traffic for
# little extra demo value and (b) the Ray Train v2 API does not yet support
# re-using a `.fit()`-ed trainer instance as a Tuner trainable. Set
# SKIP_TUNE_CELL=1 to opt out; leave unset for human local / Colab runs.
if os.environ.get("SKIP_TUNE_CELL"):
    print("Tune sweep skipped (SKIP_TUNE_CELL is set).")
else:
    from ray import tune
    from ray.tune.tuner import TuneConfig, Tuner

    tuner = Tuner(
        trainer,
        param_space={"train_loop_config": {"lr": tune.grid_search([0.001, 0.01, 0.1])}},
        tune_config=TuneConfig(num_samples=1, metric="loss", mode="min"),
    )
    tune_result = tuner.fit()
    best = tune_result.get_best_result(metric="loss", mode="min")

    print(f"\n\u2713 Sweep complete ({len(tune_result)} trials, storage: {STORAGE_KIND})")
    print(f"  Best lr:    {best.config['train_loop_config']['lr']}")
    print(f"  Best loss:  {best.metrics['loss']:.4f}")
    print(f"  Best run:   {best.path}")


## Summary

### What We Accomplished

✓ **Streamed California Housing from a public B2 bucket**, no manual download, no API key for read
✓ **Trained a PyTorch regression model with Ray Train**, distributed across two workers, one checkpoint per epoch
✓ **Ran an optional Ray Tune sweep**, three trials, automatic best-trial selection
✓ **Saved every checkpoint and trial artifact via** ``RunConfig.storage_path``, your private B2 bucket if you supplied credentials, otherwise a local tmpdir

### Your B2 Bucket Structure (after running §3 with credentials)

```
your-private-bucket/
└── ray-train-runs/
    └── california_housing_b2_demo/
        └── TorchTrainer_<run-id>/
            ├── checkpoint_000000/
            ├── checkpoint_000001/
            └── … (one per epoch, one set per Tune trial)
```

### Key Benefits of B2 for Ray Train

- **No egress fees through Cloudflare**, pull checkpoints anywhere without unexpected costs
- **S3-compatible**, works with existing pyarrow / boto3 / AWS SDK code, no Ray patches needed
- **One-bucket pattern**, datasets, checkpoints, and Tune trial artifacts can all live in B2
- **Public-read + private-write split**, distribute datasets without keys, secure your training outputs

### Next Steps

- Resume training from a saved checkpoint with ``Trainer.restore`` / ``Tuner.restore``
- Swap in your own dataset by uploading parquet to B2 (see ``prepare_dataset.py`` for the pattern) and updating ``DATASET_PREFIX``
- Scale to a multi-node cluster with ``ScalingConfig(num_workers=N, use_gpu=True)``
- Read more in the [Ray Train persistent-storage user guide](https://docs.ray.io/en/master/train/user-guides/persistent-storage.html#s3-compatible-storage-minio-backblaze-b2-etc)
- Explore other Backblaze B2 example notebooks: <https://github.com/backblaze-b2-samples/notebooks>